# 17. 홍역 93쪽 항목 4 블록 추가 (Q05 근거)

- 목적: 골든셋 Q05(MMR 2차 미완료자 학교 안내)의 근거 블록을 만든다.
- 단계: (1) 원문 대조 (2) Codex 검토 APPROVE (3) blocks.jsonl 추가 (4) 중복·검색 확인
- 블록 내용은 변환본 md 93쪽 항목 4를 그대로 쓴다. 원본 이미지의 "·"는 md에 없어 md 표기를 따른다.
- 주체는 학교[담당교사]이고, 행동은 "안내"까지만이다. 접종 시행이나 의무화로 추출하지 않는다.

In [1]:
# 1) 원문 대조: 발췌문이 93쪽 변환본에 실제로 있는지 확인한다 (읽기 전용)
import sys
sys.path.insert(0, "../../src")
from preprocess.prevention_legal.pages import excerpt_in_page

# 블록 text와 같은 문자열 (md 93쪽 항목 4)
excerpt = "4. 교내 **MMR(홍역** **유행성이하선염** **풍진) 2차 접종 미완료자** 가 있는 경우 감염 예방을 위해 **예방접종을 완료할 필요가 있음을 안내**"
print("원문 일치:", excerpt_in_page(excerpt, "measles", 93))

원문 일치: True


In [2]:
# 2) 블록 파일 확인: 새 chunk_id가 한 번만 있고, 전체 고유성이 유지되는지 확인한다
import json
from pathlib import Path

path = Path("../../data/interim/prevention_legal/blocks.jsonl")
rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
ids = [r["chunk_id"] for r in rows]
new_id = "kdca_measles_2026:p0093:b04"
print("전체 블록:", len(rows), "/ 고유 chunk_id:", len(set(ids)))
print("새 블록 개수:", ids.count(new_id))

전체 블록: 31 / 고유 chunk_id: 31
새 블록 개수: 1


In [3]:
# 3) 검색 확인: Q05 형태의 질문에서 새 블록이 상위 10개 안에 드는지 본다 (키워드 검색 기준)
from rag.prevention_legal.run import search_blocks

for q in ["교내 홍역 MMR 2차 접종 미완료자 학교 안내 해야 하나요", "MMR 2차 접종 미완료 학생 예방접종"]:
    hits = [b["chunk_id"] for b in search_blocks({"question": q, "region": "NATIONAL",
                                                  "allowed_document_ids": ["kdca_measles_2026"]}, k=10)]
    rank = hits.index(new_id) + 1 if new_id in hits else None
    print(q, "->", f"{rank}위" if rank else "상위 10 밖")

교내 홍역 MMR 2차 접종 미완료자 학교 안내 해야 하나요 -> 8위
MMR 2차 접종 미완료 학생 예방접종 -> 8위


## 결과 기록 (2026-11-05)
- 원문 대조 통과, Codex 검토 APPROVE (역할 맥락 유지, 안내까지만 추출).
- blocks.jsonl: 30 → 31 블록, chunk_id 고유성 유지.
- 검색: Q05 형태 질문에서 8위 (상위 10 안). 키워드 점수가 낮아 여유가 적다.
- 아직 하지 않은 것: run.py의 답 카드(SAMPLE_CARDS 등) 추가와 Q05 답 검증. 이 설계는 사용자 결정 후 진행한다.